# Aula de Revisão Integrada (Conteúdos das Aulas 01 a 07)

---

## Storytelling: Operação Auditoria Express — *Data Organized*

### O Contexto
Você acaba de completar o primeiro ciclo de imersão como estagiário(a) de dados na **Data Organized** (Curitiba-PR). Na manhã desta sexta-feira, o **Analista Sênior** convocou uma reunião de emergência: o diretor comercial apresentará os resultados de 2.000 contratos fechados em 2026 para o conselho de investidores, mas há suspeitas de que **valores atípicos (outliers)** estejam distorcendo o ticket médio e escondendo gargalos operacionais nas regiões.

### Sua Missão
Antes que os dados sejam liberados para o servidor de produção e para o dashboard executivo, você recebeu o arquivo `data_organized_contratos_v2000.csv` no seu *container* de homologação e precisa conduzir uma **auditoria completa em 6 etapas**, resgatando todas as ferramentas aprendidas nas **Aulas 01 a 07**:
* **Etapa 1 (Aula 01):** Diagnóstico estrutural, classificação de variáveis e tratamento de nulos com Pandas.
* **Etapa 2 (Aula 02):** Fatiamento (`loc`, `iloc`, `query`) e análise de quartis ($Q1, Q2, Q3$).
* **Etapa 3 (Aulas 03 e 04):** Simulação de consulta SQL (DQL) e cruzamento relacional (`merge` / `JOIN`) com a tabela de metas regionais.
* **Etapa 4 (Aula 05):** Conversão para `NumPy Array` e teste de confiabilidade da média (Distância Média-Mediana).
* **Etapa 5 (Aula 06):** Detecção matemática de *Outliers* via Intervalo Interquartil ($IQR$) e ordenação (`sort_values`).
* **Etapa 6 (Aula 07):** Cálculo de amplitude total e construção de painel visual 1x2 (Boxplot + Resumo Estatístico).

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Caminho dinâmico para a pasta 'dados' e o arquivo 'tabela.csv'
caminho = Path('DATA') / 'data_organized_contratos_v2000.csv'

# 1. Carregar base de dados
df = pd.read_csv(caminho, encoding='latin1')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'DATA\\data_organized_contratos_v2000.csv'

---
## Etapa 1 (Revisão Aula 01): Anatomia da Base, Variáveis e Dados Nulos

### Perguntas Norteadoras
1. Quais colunas da base são **variáveis qualitativas** e quais são **variáveis quantitativas** (discretas ou contínuas)?
2. Existe alguma coluna com valores ausentes (`NaN`)? Como uma `Series` do Pandas lida com valores nulos ao calcular a média em comparação com uma lista comum do Python?
3. Qual é o faturamento total (`sum`), o ticket médio (`mean`), o menor (`min`) e o maior (`max`) contrato registrado?

In [ ]:
import pandas as pd
from pathlib import Path

# Busca o arquivo na pasta atual e em pastas vizinhas/superiores
arquivo = 'data_organized_contratos_v2000.csv'
caminhos = list(Path('.').rglob(arquivo)) or list(Path('..').rglob(arquivo))

if caminhos:
    caminho_correto = caminhos[0]
    print(f"Arquivo encontrado em: {caminho_correto}")
    df = pd.read_csv(caminho_correto)
else:
    print("Arquivo não encontrado nas pastas próximas. Veja onde você está:")
    print("Pasta atual:", Path.cwd())


Arquivo encontrado em: ..\DATAS\data_organized_contratos_v2000.csv


---
## Etapa 2 (Revisão Aula 02): Fatiamento Estratégico e Quartis ($Q1, Q2, Q3$)

### Perguntas Norteadoras
1. Utilizando `.query()`, filtre apenas os contratos da região **'Sudeste'** na linha **'Cloud & Storage'**. Quantos contratos atendem a esse critério e qual o valor total gerado por esse nicho?
2. Utilizando `.iloc`, exiba apenas as linhas de índice `10` a `15` e as colunas `0`, `2` e `4`.
3. Calcule os quartis **$Q1$ (25%)**, **$Q2$ (50% - Mediana)** e **$Q3$ (75%)** da coluna `Valor_Contrato_BRL`. O que o valor de $Q3$ informa sobre 75% dos contratos fechados pela empresa?

In [20]:
import numpy as np

# 1. Filtro com .query()
col_regiao = next(c for c in df.columns if 'reg' in c.lower())
col_linha  = next(c for c in df.columns if 'lin' in c.lower() or 'prod' in c.lower() or 'serv' in c.lower())

filtro_nicho = df.query(f"`{col_regiao}` == 'Sudeste' and `{col_linha}` == 'Cloud & Storage'")

qtd_contratos = len(filtro_nicho)
total_nicho = filtro_nicho.get('Valor_Contrato_BRL').sum()

print(f"1. Contratos (Sudeste + Cloud & Storage): {qtd_contratos}")
print(f"   Valor Total Gerado: R$ {total_nicho:,.2f}\n")

# 2. Fatiamento por posição com .iloc (linhas 10 a 15 e colunas 0, 2 e 4)
fatiamento = df.iloc[10:16, list((0, 2, 4))]
print("2. Fatiamento com .iloc (linhas 10 a 15, colunas 0, 2 e 4):")
display(fatiamento)

# 3. Cálculo dos Quartis usando describe() e np.percentile()
print("\n3. Resumo com .describe():")
print(df.get('Valor_Contrato_BRL').describe())

valores_validos = df.get('Valor_Contrato_BRL').dropna()
q1, q2, q3 = np.percentile(valores_validos, (25, 50, 75))

print("\nQuartis com np.percentile:")
print(f"   Q1 (25%): R$ {q1:,.2f}")
print(f"   Q2 (50% - Mediana): R$ {q2:,.2f}")
print(f"   Q3 (75%): R$ {q3:,.2f}")


1. Contratos (Sudeste + Cloud & Storage): 236
   Valor Total Gerado: R$ 1,256,511.31

2. Fatiamento com .iloc (linhas 10 a 15, colunas 0, 2 e 4):


,ID_Contrato,Regiao_Cliente,Qtd_Licencas
10,TRX-10010,Sul,19
11,TRX-10011,Sul,3
12,TRX-10012,Nordeste,12
13,TRX-10013,Sul,5
14,TRX-10014,Sudeste,2
15,TRX-10015,Sudeste,22



3. Resumo com .describe():
count     2000.00000
mean      5125.50198
std       5572.70308
min         30.00000
25%       2488.69250
50%       3885.85500
75%       6038.76500
max      72500.00000
Name: Valor_Contrato_BRL, dtype: float64

Quartis com np.percentile:
   Q1 (25%): R$ 2,488.69
   Q2 (50% - Mediana): R$ 3,885.86
   Q3 (75%): R$ 6,038.77


---
## Etapa 3 (Revisão Aulas 03 e 04): Visão Relacional (SQL DQL + Cruzamento de Fontes)

O setor financeiro enviou uma planilha auxiliar com a **Meta de Ticket Médio** e o **Gerente Responsável** por cada região (simulando uma tabela `dim_regioes` vinda do MySQL).

### Perguntas Norteadoras
1. Como você escreveria a consulta **SQL (DQL)** equivalente para obter o total de vendas agrupado por `Regiao_Cliente` ordenado do maior para o menor?
2. Utilizando `pd.merge()` (equivalente ao `INNER JOIN` / `LEFT JOIN` do SQL), integre o resumo regional do DataFrame `df` com a tabela `df_gerentes` abaixo e identifique quais regiões superaram a meta de ticket médio.

In [22]:
# Tabela auxiliar simulando extração do banco MySQL (dim_regioes)
df_gerentes = pd.DataFrame({
    'Regiao_Cliente': ['Sul', 'Sudeste', 'Centro-Oeste', 'Nordeste', 'Norte'],
    'Gerente_Regional': ['Carlos Mendes', 'Fernanda Lima', 'Roberto Alves', 'Juliana Costa', 'Marcos Souza'],
    'Meta_Ticket_BRL': [4800.00, 5200.00, 4500.00, 4700.00, 4600.00]
})

# 1. Agrupamento por região (calculando Total de Vendas e Ticket Médio)
resumo_regional = (
    df.groupby('Regiao_Cliente', as_index=False)
    .agg(
        Total_Vendas_BRL=('Valor_Contrato_BRL', 'sum'),
        Ticket_Medio_BRL=('Valor_Contrato_BRL', 'mean')
    )
    .sort_values(by='Total_Vendas_BRL', ascending=False)
)

# 2. Cruzamento com a tabela de gerentes (LEFT JOIN / INNER JOIN)
df_consolidado = pd.merge(
    resumo_regional,
    df_gerentes,
    on='Regiao_Cliente',
    how='left'
)

# 3. Coluna booleana identificando se a região superou a meta de ticket médio
df_consolidado['Superou_Meta'] = df_consolidado['Ticket_Medio_BRL'] > df_consolidado['Meta_Ticket_BRL']

print("--- VISÃO CONSOLIDADA POR REGIÃO ---")
display(df_consolidado)

# Exibe apenas os nomes das regiões que bateram a meta
regioes_acima = df_consolidado.loc[df_consolidado['Superou_Meta'], 'Regiao_Cliente'].tolist()
print("Regiões que superaram a meta de ticket médio:", regioes_acima)


--- VISÃO CONSOLIDADA POR REGIÃO ---


,Regiao_Cliente,Total_Vendas_BRL,Ticket_Medio_BRL,Gerente_Regional,Meta_Ticket_BRL,Superou_Meta
0,Sudeste,4138041.90,5185.516165,Fernanda Lima,5200.0,False
1,Sul,2459653.73,4929.165792,Carlos Mendes,4800.0,True
2,Nordeste,1546549.00,5054.081699,Juliana Costa,4700.0,True
3,Centro-Oeste,1317197.19,5332.782146,Roberto Alves,4500.0,True
4,Norte,789562.14,5263.747600,Marcos Souza,4600.0,True


Regiões que superaram a meta de ticket médio: ['Sul', 'Nordeste', 'Centro-Oeste', 'Norte']


---
## Etapa 4 (Revisão Aula 05): NumPy Arrays e Confiabilidade da Média

### Perguntas Norteadoras
1. Converta a coluna `Valor_Contrato_BRL` em um **Array NumPy** e calcule a média (`np.mean`) e a mediana (`np.median`).
2. Calcule a distância relativa entre média e mediana pela fórmula do PTD: $\text{Distância} = \frac{\text{Média} - \text{Mediana}}{\text{Mediana}}$.
3. Sabendo que uma distância $\ge |25\%|$ indica **alta influência de valores extremos**, a média é uma medida confiável para o diretor apresentar ao conselho? A distorção é puxada por valores maiores ou menores?

In [ ]:
# Desenvolva aqui o código da Etapa 4 (np.array, mean, median e regra de distância):
import numpy as np

array_valores = df.get('Valor_Contrato_BRL').dropna().to_numpy()

media = np.mean(array_valores)
mediana = np.median(array_valores)

distancia_relativa = (media - mediana) / mediana
distancia_percentual = distancia_relativa * 100

print(f"1. Média (np.mean):     R$ {media:,.2f}")
print(f"   Mediana (np.median): R$ {mediana:,.2f}")
print(f"2. Distância Relativa:  {distancia_relativa:.4f} ({distancia_percentual:+.2f}%)\n")

if abs(distancia_percentual) >= 25:
    direcao = "valores MAIORES (outliers superiores / assimetria à direita)" if media > mediana else "valores MENORES (outliers inferiores / assimetria à esquerda)"
    print("3. Diagnóstico: A média NÃO é confiável (distância >= 25%).")
    print(f"   A distorção é puxada por {direcao}.")
else:
    print("3. Diagnóstico: A média É confiável (distância < 25%, baixa influência de extremos).")



1. Média (np.mean):     R$ 5,125.50
   Mediana (np.median): R$ 3,885.86
2. Distância Relativa:  0.3190 (+31.90%)

3. Diagnóstico: A média NÃO é confiável (distância >= 25%).
   A distorção é puxada por valores MAIORES (outliers superiores / assimetria à direita).


---
## Etapa 5 (Revisão Aula 06): Caça aos Outliers via IQR

### Perguntas Norteadoras
1. Calcule o **Intervalo Interquartil ($IQR = Q3 - Q1$)**, o **Limite Inferior ($Q1 - 1,5 \times IQR$)** e o **Limite Superior ($Q3 + 1,5 \times IQR$)** para `Valor_Contrato_BRL`.
2. Quantos contratos são **Outliers Superiores** e quantos são **Outliers Inferiores**?
3. Exiba os **5 maiores outliers superiores** ordenados de forma decrescente (`sort_values(ascending=False)`). A quais regiões e linhas de produto eles pertencem?

In [ ]:
# Desenvolva aqui o código da Etapa 5 (IQR, limites e filtragem/ordenação de outliers):



---
## Etapa 6 (Revisão Aula 07): Amplitude Total e Painel Executivo (1x2)

### Perguntas Norteadoras
1. Qual é a **Amplitude Total** (`max - min`) dos valores de contrato?
2. Construa um painel no Matplotlib com **1 linha e 2 colunas (`plt.subplots(1, 2)`)**, exibindo na esquerda o **Boxplot** de `Valor_Contrato_BRL` e na direita um **cartão de resumo textual (`plt.text`)** com todas as medidas calculadas (Mínimo, Limite Inferior, Q1, Mediana, Média, Q3, Limite Superior, Máximo, IQR e Amplitude).
3. **Veredito Final para o Analista Sênior:** Que recomendação você dá ao diretor comercial antes da reunião com os investidores?

In [ ]:
# Desenvolva aqui o código da Etapa 6 (Painel 1x2 com Boxplot + plt.text):

